# Robustness check — rolling temporal splits (AAPL)

Requested by the supervisor: re-run the main pipeline on different temporal splits of the
*same, already-downloaded* dataset, to check whether the central result (Black--Scholes
winning in the money, XGBoost winning out of the money) is specific to the original
split or holds more generally.

Design choices, to keep this comparable to the main analysis and cheap to run:
- **Rolling, non-overlapping windows** (not expanding): each split uses roughly the same
  amount of training data, so a difference across splits can be attributed to *which*
  period the model was trained/tested on, not to *how much* data it saw.
- **Same five inputs, same v4 target** (`log(C/K)`) as the main model.
- **Same hyperparameters** already selected via rolling-window CV on the original split
  (Table 10) — no new grid search, to keep this fast.
- **XGBoost only.** The neural network is not re-run here; if time allows, the same
  structure below can be reused for it.
- Only three non-overlapping splits fit inside the ~5 years of data actually downloaded
  (31 Aug 2020 -- 29 Aug 2025), so training windows here are shorter (~12 months) than in
  the main analysis (~3.5 years). Absolute error levels are therefore not directly
  comparable to the main results table; what is comparable is the **pattern across
  moneyness buckets** within and across these three splits.


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

import joblib


In [ ]:
aapl = pd.read_csv("data/processed/AAPL_cleaned.csv")
aapl["date"] = pd.to_datetime(aapl["date"])
aapl["exdate"] = pd.to_datetime(aapl["exdate"])

print(aapl.shape)
aapl.head()


## Shared settings (identical to the main notebook)

In [ ]:
BINS   = [0, 0.8, 0.95, 1.05, 1.2, 2, 5, 100]
LABELS = ["Deep OTM", "OTM", "ATM", "ITM", "Deep ITM", "Very Deep ITM", "Extreme ITM"]

def bucket_mae(df, preds):
    """MAE by moneyness bucket for BS and each model; last row = all options."""
    t = pd.DataFrame({"bucket": pd.cut(df["moneyness (S/K)"], bins=BINS, labels=LABELS),
                      "BS": np.abs(df["price"].values - df["bs_price"].values)})
    for k, p in preds.items():
        t[k] = np.abs(df["price"].values - np.asarray(p))
    out = t.groupby("bucket", observed=True).mean()
    out.insert(0, "n", t.groupby("bucket", observed=True).size())
    out.loc["ALL"] = [len(t)] + list(t.drop(columns="bucket").mean())
    return out.astype({"n": int})

cv_feature_cols = ["moneyness (S/K)", "T", "rate", "q", "volatility"]
cv_target_col = "price"


## 1. Black-Scholes benchmark

Identical formula and inputs as the main notebook. BS price does not depend on the
split, so it is computed once on the full dataset.


In [ ]:
d1 = (
    np.log(aapl["close"] / aapl["strike_price"])
    + (aapl["rate"] - aapl["q"] + 0.5 * aapl["volatility"] ** 2) * aapl["T"]
) / (aapl["volatility"] * np.sqrt(aapl["T"]))

d2 = d1 - aapl["volatility"] * np.sqrt(aapl["T"])

aapl["bs_price"] = (
    aapl["close"] * np.exp(-aapl["q"] * aapl["T"]) * norm.cdf(d1)
    - aapl["strike_price"] * np.exp(-aapl["rate"] * aapl["T"]) * norm.cdf(d2)
)

aapl["bs_price"].describe()


## 2. Reused hyperparameters (from Table 10, main analysis)

No new grid search: the goal is to test robustness of the *result*, not to re-tune the
model for each split.


In [ ]:
best_params = {"max_depth": 5, "learning_rate": 0.01, "n_estimators": 500, "subsample": 1.0}
best_params = {k: (int(v) if k in ["max_depth", "n_estimators"] else float(v)) for k, v in best_params.items()}
print(best_params)


## 3. Three rolling, non-overlapping splits

Same length pattern (train / validation / test) repeated three times across the sample,
sliding forward. Split A is close to, but not identical to, the split used in the main
analysis (that one used quantile cutoffs at 70%/85% of the *full* 5-year sample; here
each split is confined to its own ~20-month block so three non-overlapping splits fit).


In [ ]:
splits = {
    "C": {
        "train": ("2020-08-31", "2021-08-31"),
        "val":   ("2021-09-01", "2022-01-01"),
        "test":  ("2022-01-02", "2022-04-30"),
    },
    "B": {
        "train": ("2022-05-01", "2023-05-01"),
        "val":   ("2023-05-02", "2023-09-01"),
        "test":  ("2023-09-02", "2023-12-31"),
    },
    "A": {
        "train": ("2024-01-01", "2025-01-01"),
        "val":   ("2025-01-02", "2025-05-01"),
        "test":  ("2025-05-02", "2025-08-29"),
    },
}

for name, s in splits.items():
    print(name, s)


## 4. Fit and evaluate each split

Same recipe as the main notebook's final model: fit on train+val with `log(C/K)` as
target, evaluate on test, convert predictions back with `exp(pred) * K`.


In [ ]:
def fit_eval_split(df, split_dates, label):
    tr = df[(df["date"] >= split_dates["train"][0]) & (df["date"] <= split_dates["train"][1])]
    va = df[(df["date"] >= split_dates["val"][0])   & (df["date"] <= split_dates["val"][1])]
    te = df[(df["date"] >= split_dates["test"][0])  & (df["date"] <= split_dates["test"][1])]
    tv = pd.concat([tr, va]).sort_values("date").reset_index(drop=True)
    te = te.sort_values("date").reset_index(drop=True)

    print(f"[{label}] train {len(tr):,} | val {len(va):,} | train+val {len(tv):,} | test {len(te):,}")
    print(f"[{label}] test window: {te['date'].min()} to {te['date'].max()}")

    X_tv = tv[cv_feature_cols]
    y_tv_log = np.log(tv[cv_target_col] / tv["strike_price"])

    model = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
    model.fit(X_tv, y_tv_log)

    X_te = te[cv_feature_cols]
    test_pred = np.exp(model.predict(X_te)) * te["strike_price"].values

    return te, test_pred, model

results = {}
for name, dates in splits.items():
    te, pred, model = fit_eval_split(aapl, dates, f"AAPL-{name}")
    results[name] = {"test": te, "pred": pred, "model": model}


## 5. Per-bucket MAE, each split

In [ ]:
bucket_tables = {}
for name, r in results.items():
    bucket_tables[name] = bucket_mae(r["test"], {"XGB": r["pred"]})
    print(f"--- Split {name} ---")
    print(bucket_tables[name])
    print()


## 6. Side-by-side comparison

The check that matters: does XGBoost beat BS on out-of-the-money/at-the-money buckets,
and does BS beat XGBoost on in-the-money buckets, in *all three* splits — or only in
the original one?


In [ ]:
summary = pd.DataFrame(index=LABELS + ["ALL"])
for name, t in bucket_tables.items():
    summary[f"BS_{name}"] = t["BS"]
    summary[f"XGB_{name}"] = t["XGB"]

summary.round(3)


In [ ]:
# Same comparison expressed as a ratio (XGB error / BS error): <1 means XGB wins
ratio = pd.DataFrame(index=LABELS + ["ALL"])
for name, t in bucket_tables.items():
    ratio[name] = t["XGB"] / t["BS"]

ratio.round(3)


## 7. Save results


In [ ]:
joblib.dump(
    {"splits": splits, "best_params": best_params, "summary": summary, "ratio": ratio,
     "bucket_tables": bucket_tables},
    "robustness_aapl_results.pkl"
)
